# 🐦 Bird Species Classifier (Transfer Learning)

Trains an image classifier to identify bird species using a pretrained CNN (EfficientNet via `timm`), fine-tuned on the **Kaggle "Birds 525 Species"** dataset.

**Before running:** In Colab, go to `Runtime > Change runtime type > Hardware accelerator > GPU` (T4 is fine).

Steps:
1. Install dependencies
2. Download dataset (Kaggle API)
3. Build dataloaders
4. Define model
5. Train
6. Evaluate
7. Try it on your own photo


## 1. Install dependencies

In [ ]:
!pip install -q timm gradio kaggle


## 2. Download the dataset

This uses the Kaggle API. You need a `kaggle.json` API token:
1. Go to https://www.kaggle.com/settings → **Create New Token** → downloads `kaggle.json`
2. Run the cell below and upload that file when prompted.


In [ ]:
import os
from google.colab import files

if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("Upload your kaggle.json file:")
    uploaded = files.upload()  # upload kaggle.json here
    os.makedirs("/root/.kaggle", exist_ok=True)
    for fname in uploaded:
        os.rename(fname, "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle credentials ready.")


In [ ]:
!kaggle datasets download -d gpiosenka/100-bird-species -p /content/data --unzip


In [ ]:
import os

DATA_DIR = "/content/data"
# The Kaggle dataset unzips with train/valid/test folders directly inside DATA_DIR
print(os.listdir(DATA_DIR)[:10])
print("Train classes:", len(os.listdir(os.path.join(DATA_DIR, "train"))))


## 3. Dataloaders

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = 224
BATCH_SIZE = 64

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE * 1.14)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = datasets.ImageFolder(f"{DATA_DIR}/train", transform=train_transform)
val_dataset   = datasets.ImageFolder(f"{DATA_DIR}/valid", transform=eval_transform)
test_dataset  = datasets.ImageFolder(f"{DATA_DIR}/test",  transform=eval_transform)

assert train_dataset.classes == val_dataset.classes == test_dataset.classes

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

class_names = train_dataset.classes
num_classes = len(class_names)
print(f"Classes: {num_classes}")
print(f"Train images: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")


## 4. Model — pretrained EfficientNet with a new classification head

In [ ]:
import timm
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def build_model(num_classes, model_name="efficientnet_b0"):
    model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
    return model.to(device)

model = build_model(num_classes)


## 5. Training loop

In [ ]:
import torch.optim as optim
from tqdm.auto import tqdm

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    torch.set_grad_enabled(train)
    for images, labels in tqdm(loader, leave=False):
        images, labels = images.to(device), labels.to(device)

        if train:
            optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        if train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


In [ ]:
NUM_EPOCHS = 10
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step()

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train loss {train_loss:.4f} acc {train_acc:.4f} | "
          f"Val loss {val_loss:.4f} acc {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({"model_state": model.state_dict(), "class_names": class_names},
                    "/content/best_bird_model.pt")
        print(f"  ✓ Saved new best model (val_acc={val_acc:.4f})")


## 6. Evaluate on the test set

In [ ]:
checkpoint = torch.load("/content/best_bird_model.pt", map_location=device)
model.load_state_dict(checkpoint["model_state"])

test_loss, test_acc = run_epoch(test_loader, train=False)
print(f"Test accuracy: {test_acc:.4f}")


## 7. Try it on your own photo

Upload a bird photo and get the predicted species + confidence.


In [ ]:
from PIL import Image
import torch.nn.functional as F
from google.colab import files

def predict(image: Image.Image, topk=5):
    model.eval()
    img_t = eval_transform(image.convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(img_t)
        probs = F.softmax(logits, dim=1)[0]
    top_probs, top_idxs = probs.topk(topk)
    return [(class_names[i], float(p)) for p, i in zip(top_probs, top_idxs)]

uploaded = files.upload()
for fname in uploaded:
    img = Image.open(fname)
    display(img.resize((300, 300)))
    results = predict(img)
    print("\nTop predictions:")
    for species, prob in results:
        print(f"  {species:35s} {prob*100:5.2f}%")


## 8. (Optional) Quick web demo with Gradio

Run this cell to get a shareable link with a simple upload UI.


In [ ]:
import gradio as gr

def gradio_predict(image):
    results = predict(image, topk=5)
    return {species: prob for species, prob in results}

demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=5),
    title="Bird Species Classifier",
    description="Upload a photo of a bird to identify its species."
)
demo.launch(share=True)
